# 01. I tre paradigmi di domanda

Ci sono tre modi per fare una domanda a un decision model. Noul restituisce la probabilita che un'affermazione sia vera. Choice restituisce una probabilita per ogni opzione che dichiariamo. Score restituisce una probabilita per ogni livello di una scala ordinata.

In questo notebook costruiamo ciascun tipo, esaminiamo le risposte dichiarate e validiamo i risultati. Ogni cella di codice fa una sola piccola cosa.


## Passo 1. Noul chiede se e vero

Una domanda Noul contiene una sola affermazione. Le risposte dichiarate sono sempre `true` e `false`, quindi il modello non puo inventare una terza opzione. Partiamo da un'affermazione vera.


In [ ]:
from decision_qa import MockClient, NoulQuestion, question_labels
client = MockClient(seed=1)


In [ ]:
q_true = NoulQuestion(label='fact', statement='Paris is the capital of France.', gold=True)
print(q_true.statement)


In [ ]:
labels = question_labels(q_true)
print(labels)
assert labels == ['true', 'false']


In [ ]:
res_true = client.decision({'question': ' geography '}, [q_true])[0]
print(res_true.probabilities)


In [ ]:
assert res_true.probabilities['true'] > res_true.probabilities['false']
print('true statement ranked first')


Ora ripetiamo l'esercizio con un'affermazione falsa. Le etichette dichiarate non cambiano; cambia solo il testo dell'affermazione. Questo mostra che il paradigma e fisso mentre l'input varia.


In [ ]:
q_false = NoulQuestion(label='fact2', statement='Rome is the capital of Germany.', gold=False)
res_false = client.decision({}, [q_false])[0]
print(res_false.probabilities)


In [ ]:
assert res_false.probabilities['false'] > res_false.probabilities['true']
print('false statement ranked last')


## Passo 2. Choice dichiara le opzioni

Choice e la scelta naturale per la classificazione. Forniamo una lista di opzioni e il modello restituisce una probabilita per opzione. `gold_index` e metadato per la valutazione.


In [ ]:
from decision_qa import ChoiceQuestion
q_choice = ChoiceQuestion(label='color', options=['red', 'green', 'blue'], gold_index=1)
print(q_choice.options)


In [ ]:
choice_labels = question_labels(q_choice)
print(choice_labels)
assert choice_labels == q_choice.options


In [ ]:
res_choice = client.decision({}, [q_choice])[0]
print(res_choice.probabilities)


## Passo 3. Mescolare le opzioni senza perdere la risposta

Un modello puo essere sensibile alla posizione. Per verificarlo mescoliamo le opzioni e dobbiamo aggiornare l'indice gold. `shuffle_with_gold` restituisce sia la nuova lista sia il nuovo indice.


In [ ]:
import random
from decision_qa.distractors import shuffle_with_gold
rng = random.Random(0)
shuffled, new_gold = shuffle_with_gold(q_choice.options, q_choice.gold_index, rng)
print(shuffled, new_gold)


In [ ]:
assert shuffled[new_gold] == q_choice.options[q_choice.gold_index]
print('gold option preserved')


## Passo 4. Score usa una scala ordinata

Score differisce da Choice perche i livelli hanno un ordine. Qui `poor`, `fair` e `good` non sono intercambiabili. Li stampiamo e confermiamo che l'ordine resta invariato.


In [ ]:
from decision_qa import ScoreQuestion
q_score = ScoreQuestion(label='quality', rubric='Rate answer quality.', levels=['poor', 'fair', 'good'], gold_index=2)
print(q_score.levels)


In [ ]:
assert q_score.levels == ['poor', 'fair', 'good']
print('rubric order preserved')


In [ ]:
res_score = client.decision({}, [q_score])[0]
print(res_score.probabilities)


## Passo 5. Confidenza e astensione

Una probabilita e utile solo se sappiamo quando fidarci. Una regola semplice agisce quando la probabilita massima supera una soglia, altrimenti passa oltre. Passare oltre puo voler dire chiamare un modello piu grande o chiedere a una persona.


In [ ]:
def act(result, threshold=0.8):
    label = max(result.probabilities, key=result.probabilities.get)
    p = result.probabilities[label]
    return label, 'commit' if p >= threshold else 'escalate'


In [ ]:
print(act(res_true))


In [ ]:
action, decision = act(res_true)
assert decision in ('commit', 'escalate')
print('policy returned a valid action')


In [ ]:
print(act(res_true, threshold=0.99))


## Passo 6. Raggruppare piu domande

Una richiesta puo contenere molte domande. Il client restituisce un risultato per domanda, nello stesso ordine. Verifichiamo che l'ordine sia preservato.


In [ ]:
batch = client.decision({}, [q_true, q_choice, q_score])
print([r.question_label for r in batch])


In [ ]:
assert [r.question_label for r in batch] == ['fact', 'color', 'quality']
print('result order matches request order')


## Passo 7. Invarianza all'ordine delle opzioni

Una decisione affidabile non deve cambiare se riordiniamo le opzioni. Lo misuriamo con `option_order_invariance`. Il mock e influenzato dal gold, quindi dovrebbe essere invariante.


In [ ]:
from decision_qa.evaluate import option_order_invariance
invariant = option_order_invariance(client, {}, q_choice)
print(invariant)


In [ ]:
assert invariant is True
print('option order did not change the top answer')


## Errori comuni

Primo, non mescolare mai i livelli di una domanda Score. Secondo, non omettere mai opzioni da una domanda Choice, perche il modello valuta solo cio che dichiariamo. Terzo, definire sempre una politica di confidenza prima di usare un decision model in produzione.

Letture gratuite: `awesome-jev-prompts` contiene 51 schemi di domande sotto licenza CC0, e le doc sui concetti introducono gli stessi tre paradigmi.
